# ML03 · 邏輯迴歸與分類

用邏輯迴歸（logistic regression）把連續的線性預測轉成「機率與類別」，建立二分類（binary classification）最基本的直覺。

## 學習目標
- 理解 sigmoid 函數如何把任意實數壓成 0 到 1 之間的機率輸出。
- 能說明二分類（binary classification）與線性迴歸（linear regression）在「輸出意義」上的差別。
- 用圖直覺認識決策邊界（decision boundary）如何把平面切成兩類。
- 用直覺（非嚴格推導）理解交叉熵（cross-entropy）損失為什麼適合分類。
- 會計算與解讀準確率（accuracy），並知道它在類別不平衡時的盲點。
- 能用 sklearn 的 LogisticRegression 完成一個自造資料的二分類示範。

In [ ]:
# 概念：統一匯入與繪圖設定，讓後面每個範例可直接畫圖且中文標題不亂碼
import numpy as np
import matplotlib.pyplot as plt

# 設定中文字型，避免圖上中文變成方框（不同系統可換成手邊有的字型）
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False   # 負號正常顯示，避免被當成特殊字元

print("環境設定完成")

## 從迴歸到分類：為什麼需要機率

二分類（binary classification）要的答案只有兩種：是 / 否、屬於 / 不屬於，習慣用標籤（label）0 與 1 表示。

線性迴歸（linear regression）輸出的是任意實數，可能是 -3、也可能是 250，沒辦法直接當「是不是某一類」的答案。我們需要一個輸出落在 0 到 1、可以解讀成機率（probability）的模型。

為什麼先看它的局限：唯有先看到「直線預測會衝出 0 到 1」這個問題，才懂得後面為什麼要把輸出壓進機率區間。

In [ ]:
# 概念：分類標籤只有 0 / 1，但直線預測會超出 0 到 1，無法當機率解讀
# 造一份假資料：樓高（公尺）對應「是否為高層建築」（1 是、0 否）
heights = np.array([8, 12, 18, 25, 33, 40, 52, 60], dtype=float)
is_highrise = np.array([0, 0, 0, 1, 1, 1, 1, 1])   # 與上面每筆樓高一一對應

# 用最小平方法硬擬一條直線（樓高 -> 標籤），示範直線預測的問題
slope, intercept = np.polyfit(heights, is_highrise, deg=1)   # deg=1 即一次直線
line_pred = slope * heights + intercept                      # 直線在每個樓高的預測值

print("樓高:", heights)
print("真實標籤:", is_highrise)
print("直線預測值:", np.round(line_pred, 2))
# 注意：直線預測會出現負數或大於 1 的值，根本不能當「機率」解讀
print("預測最小值:", round(line_pred.min(), 2), " 預測最大值:", round(line_pred.max(), 2))

## sigmoid 函數與機率輸出

sigmoid 函數（sigmoid function）把任意實數（這裡稱為線性分數 z）壓進 0 到 1 之間，使輸出可以解讀成機率（probability output）。

公式：sigmoid(z) = 1 / (1 + e^(-z))

它的形狀像一條平滑的 S：z 很大時趨近 1、很小時趨近 0、z = 0 時剛好是 0.5。得到機率後，用閾值（threshold）0.5 當分界：機率大於等於 0.5 判為類別 1，否則判為類別 0。

In [ ]:
# 概念：用 sigmoid 把線性分數壓成機率，並畫出 S 形曲線建立直覺
def sigmoid(z):
    return 1 / (1 + np.exp(-z))   # 對應公式 1 / (1 + e^(-z))，z 可為陣列（向量化）

z = np.linspace(-8, 8, 200)       # -8 到 8 取 200 個等距點當輸入分數
plt.figure(figsize=(5, 3))
plt.plot(z, sigmoid(z))
plt.axhline(0.5, color="gray", linestyle="--")   # 0.5 是常用的判類閾值
plt.title("sigmoid 函數")
plt.xlabel("線性分數 z")
plt.ylabel("機率")
plt.show()

# 把自造的「樓高分數」餵進 sigmoid，看樓越高機率越接近 1
heights = np.array([8, 18, 33, 52], dtype=float)
scores = 0.15 * heights - 3.5     # 自訂一個線性分數：樓越高分數越大
probs = sigmoid(scores)           # 轉成「屬於高層」的機率
for h, p in zip(heights, probs):
    print(f"樓高 {h:4.0f} 公尺 -> 機率 {p:.3f}")

## 決策邊界：把平面切成兩類

當有兩個特徵時，每個樣本是特徵空間（feature space）裡的一個點。模型先算線性分數，再經 sigmoid 得到機率，最後用閾值 0.5 判類。

「機率剛好等於 0.5」的那條線，就是決策邊界（decision boundary）：線的一邊判為類別 1、另一邊判為類別 0。當一條直線就能把兩類大致分開時，稱資料線性可分（linearly separable）。

為什麼用圖看：在二維散佈圖上把邊界畫出來，最能直覺看到模型是怎麼「切」資料的。

In [ ]:
# 概念：兩特徵下，機率等於 0.5 的線即決策邊界，把平面切成兩類
rng = np.random.default_rng(0)   # 固定亂數種子，讓每次結果一致

# 造兩群帶標籤的點：類別 0（住宅，樓矮基地小）、類別 1（商辦，樓高基地大）
res = rng.normal(loc=[15, 80], scale=[4, 15], size=(40, 2))    # 住宅：樓高 / 基地面積
com = rng.normal(loc=[45, 180], scale=[6, 25], size=(40, 2))   # 商辦
X = np.vstack([res, com])                                      # 疊成 (80, 2) 特徵矩陣
y = np.array([0] * 40 + [1] * 40)                              # 前 40 住宅、後 40 商辦

# 自訂一條決策邊界（這裡手動給定權重示意；下一節的損失與之後的工具會自動學）
w = np.array([0.08, 0.02])   # 兩特徵的權重
b = -5.0                     # 偏置 bias

# 邊界是 w0*x0 + w1*x1 + b = 0，解出 x1 對 x0 的直線
x0_line = np.linspace(X[:, 0].min(), X[:, 0].max(), 50)
x1_line = -(w[0] * x0_line + b) / w[1]   # 由邊界方程式移項求得

plt.figure(figsize=(5, 4))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label="住宅 (0)")
plt.scatter(X[y == 1, 0], X[y == 1, 1], label="商辦 (1)")
plt.plot(x0_line, x1_line, "k--", label="決策邊界")
plt.xlabel("樓高")
plt.ylabel("基地面積")
plt.legend()
plt.title("決策邊界把平面切成兩類")
plt.show()

## 交叉熵損失的直覺

損失函數（loss function）衡量「預測有多差」，模型靠最小化它來學習。分類問題不用線性迴歸的均方誤差，而用交叉熵（cross-entropy）。

公式（單筆）：loss = -[y·log(p) + (1-y)·log(1-p)]，其中 y 是真實標籤、p 是模型給的機率。

直覺：交叉熵會放大「很有信心卻猜錯」的懲罰。當你斷言機率 0.99 卻其實是另一類，log 會給出很大的損失；猜對且有信心則損失趨近 0。這正是分類想要的：鼓勵正確且校準（confidence）的機率。

In [ ]:
# 概念：用幾組（真實標籤, 預測機率）手算交叉熵，比較猜對與猜錯的懲罰差距
def cross_entropy(y, p):
    eps = 1e-12                      # 注意：log(0) 會是負無限大，加極小值 eps 避免報錯
    p = np.clip(p, eps, 1 - eps)     # 把機率夾在 (0, 1) 開區間內
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

# 每組是 (真實標籤 y, 預測機率 p) 與情境說明
cases = [
    (1, 0.95, "猜對且有信心"),
    (1, 0.55, "猜對但沒把握"),
    (1, 0.05, "猜錯且很有信心"),
    (0, 0.02, "猜對且有信心"),
    (0, 0.97, "猜錯且很有信心"),
]
for y, p, desc in cases:
    loss = cross_entropy(y, p)
    print(f"y={y}  p={p:.2f}  loss={loss:.3f}   ({desc})")

## 用準確率評估分類

準確率（accuracy）是最直覺的分類指標：猜對的筆數除以總筆數。

公式：accuracy = 猜對筆數 / 總筆數

它好懂，但有盲點：當類別不平衡（class imbalance）時會誤導。例如 95% 的樣本都是同一類，模型「全部猜多數類」也有 95% 準確率，看似很高卻完全沒抓到少數類。所以光看 accuracy 不夠，要對不平衡保持警覺。

In [ ]:
# 概念：手算準確率，並用「全猜多數類」示範高準確率也可能沒用
# 造一組不平衡的真實標籤：18 筆類別 0、2 筆類別 1（少數類）
y_true = np.array([0] * 18 + [1] * 2)

# 模型 A：很懶，不管輸入一律猜 0（多數類）
pred_all_zero = np.zeros_like(y_true)
acc_all_zero = (pred_all_zero == y_true).mean()   # 相等的比例即準確率

# 模型 B：有抓到一個少數類，但也誤判一筆
pred_model_b = np.array([0] * 17 + [1] + [1, 0])
acc_model_b = (pred_model_b == y_true).mean()

print("全猜多數類 準確率:", round(acc_all_zero, 3))   # 高得嚇人，卻一個少數類都沒抓到
print("模型 B    準確率:", round(acc_model_b, 3))
# 注意：兩者準確率接近，但只有模型 B 真的抓到少數類；不平衡時別只看 accuracy
print("全猜多數類抓到的少數類數量:", int(((pred_all_zero == 1) & (y_true == 1)).sum()))
print("模型 B 抓到的少數類數量:", int(((pred_model_b == 1) & (y_true == 1)).sum()))

## sklearn LogisticRegression 實作示範

把前面的概念串起來：sklearn 的 LogisticRegression 會自動學出最適合的權重，最小化交叉熵。常用三個方法：

| 方法 | 作用 |
|---|---|
| fit(X, y) | 用訓練資料學出模型參數 |
| predict(X) | 輸出每筆的類別（0 / 1） |
| predict_proba(X) | 輸出每筆屬於各類的機率 |

實務上會先把資料切成訓練集（train）與測試集（test），用沒看過的測試集評估才公平；這裡概念帶過，重點在體會「自造資料到輸出機率與類別」的完整流程。

In [ ]:
# 概念：用 LogisticRegression 跑完整流程，輸出類別、機率與準確率
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(1)
# 造兩類建築資料：類別 0（住宅）、類別 1（商辦），各 60 筆、兩個特徵
res = rng.normal(loc=[15, 80], scale=[4, 15], size=(60, 2))
com = rng.normal(loc=[45, 180], scale=[6, 25], size=(60, 2))
X = np.vstack([res, com])
y = np.array([0] * 60 + [1] * 60)

# train / test split：用測試集評估才不會「背答案」
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

model = LogisticRegression()
model.fit(X_train, y_train)          # 學出權重（自動最小化交叉熵）

pred = model.predict(X_test)                 # 每筆預測類別
proba = model.predict_proba(X_test)[:, 1]    # 取「屬於類別 1」的機率（第 1 欄）
acc = (pred == y_test).mean()                # 測試集準確率

print("測試集準確率:", round(acc, 3))
print("前 5 筆  類別 / 屬於商辦的機率:")
for c, p in zip(pred[:5], proba[:5]):
    print(f"  類別 {c}   機率 {p:.3f}")

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 numpy 造（仿真即可），完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）高層建築機率小判斷（整合：sigmoid + 閾值）
#   情境：用「樓高」單一特徵，判斷一棟樓是否屬於高層建築（資料自造）。
#   要求：
#     1. 用 numpy 自造 10 筆樓高，並各算一個線性分數（例如 0.15 * 樓高 - 3.5）。
#     2. 套用 sigmoid 得到機率，再以 0.5 閾值轉成 0 / 1 類別，逐筆印出。
#   驗收：應看到每筆樓高對應的機率與最終類別，且樓越高機率越接近 1。
# TODO: 在這裡完成

In [ ]:
# TODO 2 ·（中間）住宅與商辦二分類（整合：決策邊界 + LogisticRegression + 準確率）
#   情境：用「樓高 + 容積率」兩個特徵，把建物分成住宅與商辦兩類（資料自造）。
#   要求：
#     1. 用 numpy 自造兩群帶標籤的二維資料（住宅一群、商辦一群），用 LogisticRegression 訓練。
#     2. 在散佈圖上畫出兩類點與一條決策邊界，並計算整體準確率印出。
#   驗收：應看到一條把兩群大致分開的邊界，以及一個明顯高於亂猜（遠大於 0.5）的準確率。
# TODO: 在這裡完成

In [ ]:
# TODO 3 ·（稍難）閾值與交叉熵的權衡（整合：機率輸出 + 交叉熵直覺 + 準確率 + 類別不平衡）
#   情境：在一份「是否為老舊待更新建物」的不平衡資料上做分類（多數為非待更新；資料自造）。
#   要求：
#     1. 自造一份正負類比例失衡的資料（例如九成非待更新）並訓練模型，取得每筆 predict_proba。
#     2. 用 0.3 / 0.5 / 0.7 三種閾值各自把機率轉成類別，分別算準確率與「被判為正類的數量」。
#     3. 結合交叉熵直覺說明：為何在不平衡資料上「高準確率」未必代表模型抓到少數類。
#   驗收：應看到不同閾值下準確率與正類數量的變化，並能講出在不平衡情境下只看 accuracy 的盲點。
# TODO: 在這裡完成

## 小結

- 分類問題的答案是類別（0 / 1），線性迴歸輸出的任意實數無法直接當「是不是某一類」，需要能輸出機率的模型。
- sigmoid 函數把線性分數壓進 0 到 1，使輸出可解讀成機率；常用 0.5 當判類閾值。
- 兩特徵時，「機率剛好 0.5」的那條線就是決策邊界，把特徵空間切成兩類；一條直線能分開即線性可分。
- 分類用交叉熵當損失，它對「很有信心卻猜錯」給很大懲罰，鼓勵正確且校準的機率。
- 準確率是猜對的比例，直覺好懂，但在類別不平衡時會誤導，光看 accuracy 不夠。
- sklearn 的 LogisticRegression 用 fit / predict / predict_proba 就能完成從自造資料到輸出機率與類別的完整流程。